In [1]:
import unittest
import dolfin
import numpy as np

from pgdrome.solver import PGDProblem, FD_matrices

In [2]:
# Define PGDrome problem copied from tests/integration/test_heat1D.py

def create_meshes(num_elem, ord, ranges):

    meshes = list()
    Vs = list()

    dim = len(num_elem)

    for i in range(dim):
        mesh_tmp = dolfin.IntervalMesh(num_elem[i], ranges[i][0], ranges[i][1])
        Vs_tmp = dolfin.FunctionSpace(mesh_tmp, "CG", ord[i])

        meshes.append(mesh_tmp)
        Vs.append(Vs_tmp)

    return meshes, Vs


def create_bc(Vs, dom, param):
    # boundary conditions list

    # Initial condition
    def init(x, on_boundary):
        return x < 0.0 + 1e-5

    initCond = dolfin.DirichletBC(Vs[1], 0, init)

    return [0, initCond, 0]


def problem_assemble_lhs_FEM(fct_F, var_F, Fs, meshes, dom, param, typ, dim):
    # problem discription left hand side of DGL for each fixed point problem

    if typ == "r":
        a = dolfin.Constant(
            dolfin.assemble(Fs[1].dx(0) * Fs[1] * dolfin.dx(meshes[1]))
            * dolfin.assemble(Fs[2] * Fs[2] * dolfin.dx(meshes[2]))
        ) * param["rho"] * param["cp"] * fct_F * var_F * dolfin.dx(
            meshes[0]
        ) + dolfin.Constant(
            dolfin.assemble(Fs[1] * Fs[1] * dolfin.dx(meshes[1]))
            * dolfin.assemble(Fs[2] * Fs[2] * dolfin.dx(meshes[2]))
        ) * param[
            "k"
        ] * fct_F.dx(
            0
        ) * var_F.dx(
            0
        ) * dolfin.dx(
            meshes[0]
        )
    if typ == "s":
        a = dolfin.Constant(
            dolfin.assemble(Fs[0] * Fs[0] * dolfin.dx(meshes[0]))
            * dolfin.assemble(Fs[2] * Fs[2] * dolfin.dx(meshes[2]))
        ) * param["rho"] * param["cp"] * fct_F.dx(0) * var_F * dolfin.dx(
            meshes[1]
        ) + dolfin.Constant(
            dolfin.assemble(Fs[0].dx(0) * Fs[0].dx(0) * dolfin.dx(meshes[0]))
            * dolfin.assemble(Fs[2] * Fs[2] * dolfin.dx(meshes[2]))
        ) * param[
            "k"
        ] * fct_F * var_F * dolfin.dx(
            meshes[1]
        )
    if typ == "w":
        a = dolfin.Constant(
            dolfin.assemble(Fs[0] * Fs[0] * dolfin.dx(meshes[0]))
            * dolfin.assemble(Fs[1].dx(0) * Fs[1] * dolfin.dx(meshes[1]))
        ) * param["rho"] * param["cp"] * fct_F * var_F * dolfin.dx(
            meshes[2]
        ) + dolfin.Constant(
            dolfin.assemble(Fs[0].dx(0) * Fs[0].dx(0) * dolfin.dx(meshes[0]))
            * dolfin.assemble(Fs[1] * Fs[1] * dolfin.dx(meshes[1]))
        ) * param[
            "k"
        ] * fct_F * var_F * dolfin.dx(
            meshes[2]
        )
    return a


def problem_assemble_rhs_FEM(
    fct_F, var_F, Fs, meshes, dom, param, Q, PGD_func, typ, nE, dim
):
    # problem discription right hand side of DGL for each fixed point problem

    IC = [param["IC_x"], param["IC_t"], param["IC_q"]]

    if typ == "r":
        l = (
            dolfin.Constant(
                dolfin.assemble(Q[1] * Fs[1] * dolfin.dx(meshes[1]))
                * dolfin.assemble(Q[2] * Fs[2] * dolfin.dx(meshes[2]))
            )
            * Q[0]
            * var_F
            * dolfin.dx(meshes[0])
            - dolfin.Constant(
                dolfin.assemble(IC[1].dx(0) * Fs[1] * dolfin.dx(meshes[1]))
                * dolfin.assemble(IC[2] * Fs[2] * dolfin.dx(meshes[2]))
            )
            * param["rho"]
            * param["cp"]
            * IC[0]
            * var_F
            * dolfin.dx(meshes[0])
            - dolfin.Constant(
                dolfin.assemble(IC[1] * Fs[1] * dolfin.dx(meshes[1]))
                * dolfin.assemble(IC[2] * Fs[2] * dolfin.dx(meshes[2]))
            )
            * param["k"]
            * IC[0].dx(0)
            * var_F.dx(0)
            * dolfin.dx(meshes[0])
        )
        if nE > 0:
            for old in range(nE):
                l += -dolfin.Constant(
                    dolfin.assemble(
                        PGD_func[1][old].dx(0) * Fs[1] * dolfin.dx(meshes[1])
                    )
                    * dolfin.assemble(PGD_func[2][old] * Fs[2] * dolfin.dx(meshes[2]))
                ) * param["rho"] * param["cp"] * PGD_func[0][old] * var_F * dolfin.dx(
                    meshes[0]
                ) - dolfin.Constant(
                    dolfin.assemble(PGD_func[1][old] * Fs[1] * dolfin.dx(meshes[1]))
                    * dolfin.assemble(PGD_func[2][old] * Fs[2] * dolfin.dx(meshes[2]))
                ) * param[
                    "k"
                ] * PGD_func[
                    0
                ][
                    old
                ].dx(
                    0
                ) * var_F.dx(
                    0
                ) * dolfin.dx(
                    meshes[0]
                )
    if typ == "s":
        l = (
            dolfin.Constant(
                dolfin.assemble(Q[0] * Fs[0] * dolfin.dx(meshes[0]))
                * dolfin.assemble(Q[2] * Fs[2] * dolfin.dx(meshes[2]))
            )
            * Q[1]
            * var_F
            * dolfin.dx(meshes[1])
            - dolfin.Constant(
                dolfin.assemble(IC[0] * Fs[0] * dolfin.dx(meshes[0]))
                * dolfin.assemble(IC[2] * Fs[2] * dolfin.dx(meshes[2]))
            )
            * param["rho"]
            * param["cp"]
            * IC[1].dx(0)
            * var_F
            * dolfin.dx(meshes[1])
            - dolfin.Constant(
                dolfin.assemble(IC[0].dx(0) * Fs[0].dx(0) * dolfin.dx(meshes[0]))
                * dolfin.assemble(IC[2] * Fs[2] * dolfin.dx(meshes[2]))
            )
            * param["k"]
            * IC[1]
            * var_F
            * dolfin.dx(meshes[1])
        )
        if nE > 0:
            for old in range(nE):
                l += -dolfin.Constant(
                    dolfin.assemble(PGD_func[0][old] * Fs[0] * dolfin.dx(meshes[0]))
                    * dolfin.assemble(PGD_func[2][old] * Fs[2] * dolfin.dx(meshes[2]))
                ) * param["rho"] * param["cp"] * PGD_func[1][old].dx(
                    0
                ) * var_F * dolfin.dx(
                    meshes[1]
                ) - dolfin.Constant(
                    dolfin.assemble(
                        PGD_func[0][old].dx(0) * Fs[0].dx(0) * dolfin.dx(meshes[0])
                    )
                    * dolfin.assemble(PGD_func[2][old] * Fs[2] * dolfin.dx(meshes[2]))
                ) * param[
                    "k"
                ] * PGD_func[
                    1
                ][
                    old
                ] * var_F * dolfin.dx(
                    meshes[1]
                )
    if typ == "w":
        l = (
            dolfin.Constant(
                dolfin.assemble(Q[0] * Fs[0] * dolfin.dx(meshes[0]))
                * dolfin.assemble(Q[1] * Fs[1] * dolfin.dx(meshes[1]))
            )
            * Q[2]
            * var_F
            * dolfin.dx(meshes[2])
            - dolfin.Constant(
                dolfin.assemble(IC[0] * Fs[0] * dolfin.dx(meshes[0]))
                * dolfin.assemble(IC[1].dx(0) * Fs[1] * dolfin.dx(meshes[1]))
            )
            * param["rho"]
            * param["cp"]
            * IC[2]
            * var_F
            * dolfin.dx(meshes[2])
            - dolfin.Constant(
                dolfin.assemble(IC[0].dx(0) * Fs[0].dx(0) * dolfin.dx(meshes[0]))
                * dolfin.assemble(IC[1] * Fs[1] * dolfin.dx(meshes[1]))
            )
            * param["k"]
            * IC[2]
            * var_F
            * dolfin.dx(meshes[2])
        )
        if nE > 0:
            for old in range(nE):
                l += -dolfin.Constant(
                    dolfin.assemble(PGD_func[0][old] * Fs[0] * dolfin.dx(meshes[0]))
                    * dolfin.assemble(
                        PGD_func[1][old].dx(0) * Fs[1] * dolfin.dx(meshes[1])
                    )
                ) * param["rho"] * param["cp"] * PGD_func[2][old] * var_F * dolfin.dx(
                    meshes[2]
                ) - dolfin.Constant(
                    dolfin.assemble(
                        PGD_func[0][old].dx(0) * Fs[0].dx(0) * dolfin.dx(meshes[0])
                    )
                    * dolfin.assemble(PGD_func[1][old] * Fs[1] * dolfin.dx(meshes[1]))
                ) * param[
                    "k"
                ] * PGD_func[
                    2
                ][
                    old
                ] * var_F * dolfin.dx(
                    meshes[2]
                )
    return l

In [3]:
def create_PGD(param={}, vs=[], q=None, _type=None):

    # define nonhomogeneous dirichlet IC
    param.update({"IC_x": dolfin.interpolate(param["IC_x"], vs[0])})
    param.update({"IC_t": dolfin.interpolate(param["IC_t"], vs[1])})
    param.update({"IC_q": dolfin.interpolate(param["IC_q"], vs[2])})

    # define heat source in x, t and q
    q_x = dolfin.interpolate(q, vs[0])
    q_t = dolfin.interpolate(dolfin.Expression("1.0", degree=1), vs[1])
    q_q = dolfin.interpolate(dolfin.Expression("x[0]*Q", Q=param["Q"], degree=1), vs[2])

    if _type == "FEM":
        ass_rhs = problem_assemble_rhs_FEM
        ass_lhs = problem_assemble_lhs_FEM
        solve_modes = ["FEM", "FEM", "FEM"]


    pgd_prob = PGDProblem(
        name="1DHeatEqu-PGD-XTQ",
        name_coord=["X", "T", "Q"],
        modes_info=["T", "Node", "Scalar"],
        Vs=vs,
        dom=0,
        bc_fct=create_bc,
        load=[q_x, q_t, q_q],
        param=param,
        rhs_fct=ass_rhs,
        lhs_fct=ass_lhs,
        probs=["r", "s", "w"],
        seq_fp=np.arange(len(vs)),
        PGD_nmax=20,
    )

    pgd_prob.stop_fp = "norm"
    pgd_prob.max_fp_it = 50
    pgd_prob.tol_fp_it = 1e-5
    # pgd_prob.fp_init = 'randomized'
    pgd_prob.norm_modes = "stiff"
    pgd_prob.PGD_tol = 1e-5  # 1e-9 as stopping criterion

    pgd_prob.solve_PGD(_problem="linear", solve_modes=solve_modes)

    print(pgd_prob.simulation_info)
    print("PGD Amplitude", pgd_prob.amplitude)

    pgd_s = pgd_prob.return_PGD()  # as PGD class instance

    return pgd_s, param

In [22]:
# define parameters
param = {
    "rho": 1,
    "cp": 1,
    "k": 0.5,
    "Tamb": 25,
    "Q": 1,
    "af": 0.2,
    "ar": 0.2,
    "xc": 0.5,
    "lx": 1,
    "lt": 1,
}  # -comparable matlab code proofed (coarse mesh)

ranges = [
    [0.0, param["lx"]],  # xmin, xmax
    [0.0, param["lt"]],  # tmin, tmax
    [0.5, 1.0],
]  # qmin, qmax

ords = [1, 1, 1]  # x, t, q
elems = [15, 10, 10]

# case heating
ff = (
    6
    * np.sqrt(3)
    / (
        (param["af"] + param["ar"])
        * param["af"]
        * param["af"]
        * np.pi ** (3 / 2)
    )
)
q = dolfin.Expression(
    "ff* exp(-3*(pow(x[0]-xc,2)/pow(af,2)))",
    degree=4,
    ff=ff,
    af=param["af"],
    ar=param["ar"],
    xc=param["xc"],
)

param["Tamb_fct"] = dolfin.Expression(
    "Tamb", degree=1, Tamb=param["Tamb"]
)  # initial condition FEM
param["IC_t"] = param["Tamb_fct"]
param["IC_x"] = dolfin.Expression("1.0", degree=1)
param["IC_q"] = dolfin.Expression("1.0", degree=1)

# MESH
meshes, vs = create_meshes(elems, ords, ranges)




In [23]:
# create PGD
pgd_fem, param = create_PGD(param=param, vs=vs, q=q, _type="FEM")

Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational p

ERROR: fix point iteration in maximum number of iterations NOT converged (enrichment loop 1) (error 3.262121e-01)


Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational p

ERROR: fix point iteration in maximum number of iterations NOT converged (enrichment loop 2) (error 5.113794e-01)


Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational p

ERROR: fix point iteration in maximum number of iterations NOT converged (enrichment loop 18) (error 9.383970e-03)


Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
Solving linear variational problem.
PGD solver option: PGD_nmax 20 / PGD tolerance 1e-10 and max FP iterations 50 and FP tolerance 1e-05; 
-- residuum norm: 7.398197866658527 --
enrichment step 0 fixed point iteration converged in 6 / error: 7.380393e-06 
-- residuum norm: 0.9158317915891236 --
<<<enrichment step 1 fixed point iteration NOT converged in 50 / error: 3.262121e-01 >>>
-- residuum norm: 0.7294008947514699 --
<<<enrichment step 2 fixed point it

In [25]:
param


{'rho': 1,
 'cp': 1,
 'k': 0.5,
 'Tamb': 25,
 'Q': 1,
 'af': 0.2,
 'ar': 0.2,
 'xc': 0.5,
 'lx': 1,
 'lt': 1,
 'Tamb_fct': Coefficient(FunctionSpace(None, FiniteElement('Lagrange', None, 1)), 159661),
 'IC_t': Coefficient(FunctionSpace(Mesh(VectorElement(FiniteElement('Lagrange', interval, 1), dim=1), 159669), FiniteElement('Lagrange', interval, 1)), 159682),
 'IC_x': Coefficient(FunctionSpace(Mesh(VectorElement(FiniteElement('Lagrange', interval, 1), dim=1), 159664), FiniteElement('Lagrange', interval, 1)), 159679),
 'IC_q': Coefficient(FunctionSpace(Mesh(VectorElement(FiniteElement('Lagrange', interval, 1), dim=1), 159674), FiniteElement('Lagrange', interval, 1)), 159685)}

In [27]:
# evaluation of pgd solution

# evaluation parameters
fixed_dim = 0
t_fixed = 0.9 * param["lt"]
q_fixed = 1.0
x_fixed = 0.5 * param["lx"]

upgd_fem = pgd_fem.evaluate(
            fixed_dim, [1, 2], [t_fixed, q_fixed], 0
        )
upgd_fem_bc = upgd_fem.compute_vertex_values()[:] + param[
            "IC_x"
        ].compute_vertex_values()[:] * param["IC_t"](t_fixed) * param["IC_q"](
            q_fixed
        )
upgd_fem_bc

array([44.6613684 , 44.76772901, 45.08671769, 45.61786342, 46.35729711,
       47.27499868, 48.23363595, 48.89733596, 48.89733596, 48.23363595,
       47.27499868, 46.35729711, 45.61786342, 45.08671769, 44.76772901,
       44.6613684 ])

In [29]:
# compute FEM residuum for given solution
print(pgd_fem.used_numModes)
for i in range(1,pgd_fem.numModes):
    print(i)
    pgd_fem.used_numModes = i
    upgd_fem = pgd_fem.evaluate(
            fixed_dim, [1, 2], [t_fixed, q_fixed], 0
        )
    upgd_fem_bc = upgd_fem.compute_vertex_values()[:] + param[
            "IC_x"
        ].compute_vertex_values()[:] * param["IC_t"](t_fixed) * param["IC_q"](
            q_fixed
        )
    print(upgd_fem_bc.max())



1
1
50.25520132838964
2
49.25077255788862
3
49.65236432621728
4
48.940810055829445
5
48.8606881224567
6
48.92140835367526
7
48.88788703054885
8
48.90931261839669
9
48.89499376880233
10
48.905578536440785
11
48.89739938867599
12
48.903971470565374
13
48.89847989759649
14
48.9031107250222
15
48.89906419332026
16
48.902589218327456
17
48.92624774550096
18
48.9042149673207
19
48.91189092909889


In [35]:
# FEM plus residuum computation using FEM or PGD solution of current time step

mesh = vs[0].mesh()
time_mesh = vs[1].mesh().coordinates()[:]

T_n = dolfin.interpolate(param["Tamb_fct"], vs[0]) #initial condition

T = dolfin.TrialFunction(vs[0])
v = dolfin.TestFunction(vs[0])
dt = dolfin.Constant(1.0)
Q = dolfin.Constant(1.0)
Q.assign(q_fixed * param["Q"])

F = (
            param["rho"] * param["cp"] * T * v * dolfin.dx()
            + dt
            * param["k"]
            * dolfin.dot(dolfin.grad(T), dolfin.grad(v))
            * dolfin.dx()
            - (
                dt * Q * q
                + param["rho"] * param["cp"] * T_n
            )
            * v
            * dolfin.dx()
        )

def a(_T,_V,_dt,_param):
    return _param["rho"] * _param["cp"] * _T * _V * dolfin.dx() + _dt * _param["k"] * dolfin.dot(dolfin.grad(_T), dolfin.grad(_V)) * dolfin.dx()

def l(_T,_V,_dt,_Q,_q,_T_n,_param):
    return (_dt * _Q * _q + _param["rho"] * _param["cp"] * _T_n) * _V * dolfin.dx()

# Time-stepping
Ttime = []
Ttmp = dolfin.Function(vs[0])
Ttmp.vector()[:] = 1 * T_n.vector()[:]
Ttime.append(Ttmp)  # otherwise it will be overwritten with new solution

TFEM = dolfin.Function(vs[0])
for i in range(len(time_mesh) - 1):
    dt.assign(time_mesh[i + 1] - time_mesh[i])
    # Compute solution
    # a, L = dolfin.lhs(F), dolfin.rhs(F)
    # dolfin.solve(a == L, T)

    dolfin.solve(a(T,v,dt,param) == l(T,v,dt,Q,q,T_n,param), TFEM)
  

    # compute residual for given solution T

    v = dolfin.TestFunction(vs[0])
    res = dolfin.assemble(a(TFEM,v,dt,param) - l(TFEM,v,dt,Q,q,T_n,param))
    # for bc in boundaries: #if there are Dirichlet boundary conditions
    #   bc.apply(r, TFEM.vector())
    print('\n',i,'residuum FEM', res.norm("l2"))

    # load PGD solution for that time step and compute residual
    print('evaluate pgd at time', time_mesh[i+1], 'q', q_fixed)
    TPGD = dolfin.Function(vs[0])
    res_pgd_modes = []
    for j in range(1,pgd_fem.numModes):
        pgd_fem.used_numModes = j
        pgd = pgd_fem.evaluate(
            0, [1, 2], [time_mesh[i+1], q_fixed], 0
            )
        
        pgd_bc = pgd.vector()[:] + param[
            "IC_x"
        ].vector()[:] * param["IC_t"](time_mesh[i+1]) * param["IC_q"](
            q_fixed
        )
        
        #TPGD = dolfin.interpolate(pgd_bc, vs[0]) 
        TPGD.vector()[:] = pgd_bc[:]

        v = dolfin.TestFunction(vs[0])
        res = dolfin.assemble(a(TPGD,v,dt,param) - l(TPGD,v,dt,Q,q,T_n,param))
        res_pgd_modes.append(res.norm("l2"))
    print('\n',i,'residuum PGD', res_pgd_modes)

    print('compare PGD and FEM', (TFEM.vector()[:] - TPGD.vector()[:]).max())

    # Update previous solution
    T_n.assign(TFEM)






Solving linear variational problem.
 0 residuum FEM 1.2381889906725338e-14
evaluate pgd at time [0.1] q 1.0

 0 residuum PGD [0.7421097068700351, 0.725916755887361, 0.7240365288265079, 0.2482686898421307, 0.24641666255719122, 0.24529096136595765, 0.24621607787649655, 0.24606288782470132, 0.24660224247808352, 0.2465232816480911, 0.24685610182031417, 0.24679128870996414, 0.24702070059565262, 0.24696260636610057, 0.24713326352410803, 0.24708038996935408, 0.25551267033947045, 0.25545237978672886, 0.25623613172333787]
compare PGD and FEM 0.36374167513812594

 1 residuum FEM 1.3982317158026115e-14
evaluate pgd at time [0.2] q 1.0
Solving linear variational problem.

 1 residuum PGD [0.7958452743915669, 0.765699924649381, 0.7608797465147236, 0.0908125460021734, 0.0898395349296595, 0.08974869496717092, 0.09046896397562175, 0.09058379093475546, 0.09084425947454997, 0.090886914836379, 0.09101269084255194, 0.0910294958153473, 0.09110376483156973, 0.09111010339761243, 0.0911596730004786, 0.091161

In [46]:
# consider only PGD solution is known - compute residuum for a specific time step


# evaluation of pgd solution at time t_fixed and q_fixed

upgd_fem = pgd_fem.evaluate(
            fixed_dim, [1, 2], [t_fixed, q_fixed], 0
        )
upgd_fem_bc_tfixed = upgd_fem.vector()[:] + param[
            "IC_x"
        ].vector()[:] * param["IC_t"](t_fixed) * param["IC_q"](
            q_fixed
        )

print('PGD solution at t:', t_fixed, upgd_fem_bc_tfixed.max())

#evaluate PGD solution for t-dt
dt = param["lt"]/elems[1]
upgd_fem = pgd_fem.evaluate(
            fixed_dim, [1, 2], [t_fixed-dt, q_fixed], 0
        )
upgd_fem_bc_tn = upgd_fem.vector()[:] + param[
            "IC_x"
        ].vector()[:] * param["IC_t"](t_fixed-dt) * param["IC_q"](
            q_fixed
        )

print('PGD solution at t-dt:', t_fixed-dt,upgd_fem_bc_tn.max())

# compute residuum:

TPGD = dolfin.Function(vs[0])
TPGD.vector()[:] = upgd_fem_bc_tfixed[:]

TPGDn = dolfin.Function(vs[0])
TPGDn.vector()[:] = upgd_fem_bc_tn[:]


v = dolfin.TestFunction(vs[0])
res = dolfin.assemble(a(TPGD,v,dt,param) - l(TPGD,v,dt,Q,q,TPGDn,param))
res_pgd_modes.append(res.norm("l2"))

print("residuum PGD", res.norm("l2"))

PGD solution at t: 0.9 48.91189092909889
PGD solution at t-dt: 0.8 46.52092563837563
residuum PGD 0.04824446405607392
